# Evaluasi Ilmiah Sistem Temu Kembali Arsip Berbasis Hasil Agregat
### Analisis Kinerja Fuzzy Matching & Normalisasi Metadata Tanpa Ekspor Teks Arsip
**Versi Notebook:** 1.0.0 (Spesifikasi Resmi Konsumsi Bundle)

---

### 🔒 Pernyataan Privasi & Integritas Data
- **Zero Raw-Data Ingestion:** Notebook ini **HANYA** membaca data hasil evaluasi teragregasi dari berkas bundle (`run_meta.json`, `per_query_results.csv`, `tuning_results.csv`, `audit_summary.json`, `agreement_summary.json`).
- **Perlindungan Privasi Dinas:** Tidak ada teks ringkasan arsip mentah (`summary_raw`), nomor SPM, nama rekanan/pihak ketiga (`party_name`), maupun teks kueri asli (`query_text`) yang diunggah ke runtime ini.
- **Validasi Keterulangan:** Seluruh berkas bundle diverifikasi terhadap hash kriptografis `manifest.sha256` yang dihasilkan langsung dari laptop peneliti dalam status *working tree* bersih.

In [ ]:
# =====================================================================
# SEL 1: PERSIAPAN LINGKUNGAN & KUNCI SEED ACAK
# Syarat lulus: Paket ilmiah terpasang dan seed acak diatur = 42
# =====================================================================
import os
import sys
import json
import hashlib
from pathlib import Path

# Pasang dependensi jika berjalan di Google Colab
try:
    import google.colab
    IN_COLAB = True
    !pip install -q pandas numpy scipy matplotlib rapidfuzz
except ImportError:
    IN_COLAB = False

import pandas as pd
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("=== VERSI LINGKUNGAN RUNTIME ===")
print(f"Python     : {sys.version.split()[0]}")
print(f"Pandas     : {pd.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"SciPy      : {scipy.__version__}")
print(f"Matplotlib : {matplotlib.__version__}")
print(f"Random Seed: {RANDOM_SEED} (terkunci)")
print("\n[STATUS] Sel 1 Lulus: Lingkungan siap.")

In [ ]:
# =====================================================================
# SEL 2: PEMUATAN BERKAS BUNDLE & PEMERIKSAAN DAFTAR PUTIH KOLOM
# Syarat lulus: Tolak bila ditemukan kolom teks arsip/terlarang
# =====================================================================
FORBIDDEN_COLUMNS = {
    "query_text", "summary_raw", "summary_clean", "party_name",
    "spm_number", "sp2d_number", "amount_rp", "haystack", "nominal"
}

EXPECTED_FILES = [
    "run_meta.json",
    "per_query_results.csv",
    "tuning_results.csv",
    "audit_summary.json",
    "agreement_summary.json",
    "manifest.sha256"
]

# Tentukan lokasi direktori bundle
BUNDLE_DIR = Path("output/colab_bundle")
if not BUNDLE_DIR.is_dir() and Path("colab_bundle").is_dir():
    BUNDLE_DIR = Path("colab_bundle")

# Jika belum ada dan berada di Google Colab, sediakan dialog unggah
if not (BUNDLE_DIR / "manifest.sha256").is_file():
    if IN_COLAB:
        from google.colab import files
        print("Silakan unggah 6 berkas bundle dari folder output/colab_bundle/:")
        uploaded = files.upload()
        BUNDLE_DIR = Path(".")
    else:
        raise FileNotFoundError(
            f"Direktori bundle {BUNDLE_DIR} tidak ditemukan. "
            "Jalankan `flask report export-bundle` terlebih dahulu di laptop."
        )

# Verifikasi keberadaan semua berkas wajib
for fname in EXPECTED_FILES:
    fpath = BUNDLE_DIR / fname
    if not fpath.is_file():
        raise FileNotFoundError(f"[GAGAL] Berkas bundle wajib tidak ditemukan: {fname}")

# Pemeriksaan keamanan kolom pada per_query_results.csv
df_per_query = pd.read_csv(BUNDLE_DIR / "per_query_results.csv")
found_forbidden = FORBIDDEN_COLUMNS.intersection(set(df_per_query.columns))
if found_forbidden:
    raise ValueError(f"[GAGAL TOTAL] Ditemukan kolom teks arsip terlarang: {found_forbidden}")

# Pemeriksaan keamanan kolom pada tuning_results.csv
df_tuning = pd.read_csv(BUNDLE_DIR / "tuning_results.csv")
found_forbidden_tuning = FORBIDDEN_COLUMNS.intersection(set(df_tuning.columns))
if found_forbidden_tuning:
    raise ValueError(f"[GAGAL TOTAL] Ditemukan kolom terlarang di tuning_results.csv: {found_forbidden_tuning}")

print(f"[OK] Seluruh {len(EXPECTED_FILES)} berkas bundle tersedia di: {BUNDLE_DIR}")
print(f"[OK] Pemeriksaan privasi lolos: 0 kolom teks arsip terlarang terdeteksi.")
print(f"     Kolom per_query_results: {list(df_per_query.columns)}")

In [ ]:
# =====================================================================
# SEL 3: VERIFIKASI INTEGRITAS KRIPTOGRAFIS & MANIFEST REPRODUCIBILITY
# Syarat lulus: tree_clean == True, mode == 'final', seluruh hash cocok
# =====================================================================
manifest_path = BUNDLE_DIR / "manifest.sha256"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest_lines = [line.strip() for line in f if line.strip()]

print("Memverifikasi Checksum SHA-256...")
for line in manifest_lines:
    expected_hash, fname = line.split(maxsplit=1)
    target_file = BUNDLE_DIR / fname
    h = hashlib.sha256()
    with open(target_file, "rb") as bf:
        while chunk := bf.read(65536):
            h.update(chunk)
    actual_hash = h.hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f"[GAGAL INTEGRITAS] Hash berkas {fname} TIDAK COCOK!")
    print(f"  [MATCH] {fname} ({actual_hash[:16]}...)")

# Baca dan validasi run_meta.json
with open(BUNDLE_DIR / "run_meta.json", "r", encoding="utf-8") as f:
    run_meta = json.load(f)

print("\n=== METADATA KETERULANGAN (RUN_META) ===")
print(f"Commit SHA       : {run_meta.get('commit_sha')}")
print(f"Status Git Tree  : {'Clean (Bersih)' if run_meta.get('tree_clean') else 'Dirty (Ada modifikasi)'}")
print(f"Mode Evaluasi    : {run_meta.get('mode')}")
print(f"Hash Data Arsip  : {run_meta.get('data_hash')[:24]}...")
print(f"Hash Set Uji     : {run_meta.get('testset_hash')[:24]}...")
print(f"Versi Ekstraktor : {run_meta.get('extractor_version')}")
print(f"Perangkat Laptop : {run_meta.get('laptop_specs', {}).get('os')}")

if run_meta.get("mode") != "final":
    raise ValueError(f"[GAGAL] Mode evaluasi harus 'final', didapatkan: {run_meta.get('mode')}")
if not run_meta.get("tree_clean"):
    print("[PERINGATAN]: Working tree berstatus kotor saat ekspor dilakukan.")

print("\n[STATUS] Sel 3 Lulus: Integritas kriptografis dan metadata terverifikasi valid.")

In [ ]:
# =====================================================================
# SEL 4: UJI KEWAJARAN DATA (SANITY CHECKS)
# Syarat lulus: Tidak ada null, metode tidak identik, metrik tidak semua nol
# =====================================================================
# 1. Periksa missing value (NaN/Null)
null_counts = df_per_query.isnull().sum()
if null_counts.any():
    raise ValueError(f"[GAGAL] Terdapat nilai null pada per_query_results:\n{null_counts[null_counts > 0]}")

# 2. Periksa jumlah kueri per split
split_counts = df_per_query.groupby('split')['query_id'].nunique()
print("Distribusi Kueri per Split:")
for split_name, count in split_counts.items():
    print(f"  - {split_name}: {count} kueri")

# 3. Periksa kelengkapan metode
methods_in_data = df_per_query[['method', 'config']].drop_duplicates()
print(f"\nJumlah Konfigurasi Metode Dievaluasi: {len(methods_in_data)}")

# 4. Uji non-identik antar metode (seluruh metrik per kueri tidak identik 100%)
metrics_cols = ['p5', 'r10', 'rr10', 'hit5', 'hit10', 'zero_result']
method_configs = df_per_query[['method', 'config']].drop_duplicates().values
for i in range(len(method_configs)):
    for j in range(i + 1, len(method_configs)):
        m1, c1 = method_configs[i]
        m2, c2 = method_configs[j]
        v1 = df_per_query[(df_per_query['method'] == m1) & (df_per_query['config'] == c1)].sort_values('query_id')[metrics_cols].values
        v2 = df_per_query[(df_per_query['method'] == m2) & (df_per_query['config'] == c2)].sort_values('query_id')[metrics_cols].values
        if np.array_equal(v1, v2):
            raise ValueError(f"[GAGAL] Metode {m1} ({c1}) dan {m2} ({c2}) menghasilkan seluruh metrik per kueri identik sepenuhnya!")

# 5. Uji metrik tidak semua nol
if df_per_query['p5'].sum() == 0.0 or df_per_query['rr10'].sum() == 0.0:
    raise ValueError("[GAGAL] Seluruh metrik bernilai nol, pipeline evaluasi tidak berjalan semestinya.")

print("\n[STATUS] Sel 4 Lulus: Uji kewajaran berhasil tanpa anomali data.")

In [ ]:
# =====================================================================
# SEL 5: TABEL UTAMA EVALUASI METRIK TEMU KEMBALI (KELOMPOK TEST SAJA)
# Menghitung P@5, R@10, MRR@10, Hit Rate, Nol Hasil, Latensi + Bootstrap 95% CI
# =====================================================================
df_test = df_per_query[df_per_query['split'] == 'test'].copy()
n_test_queries = df_test['query_id'].nunique()

def calc_bootstrap_ci(series, n_boot=2000, ci=0.95, seed=42):
    """Hitung interval kepercayaan 95% dengan bootstrap persentil."""
    rng = np.random.default_rng(seed)
    vals = np.array(series, dtype=float)
    boot_means = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    low = np.percentile(boot_means, (1 - ci) / 2 * 100)
    high = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return round(low, 4), round(high, 4)

main_table_rows = []
grouped = df_test.groupby(['method', 'config'])

for (m, cfg), grp in grouped:
    p5_mean = grp['p5'].mean()
    p5_ci = calc_bootstrap_ci(grp['p5'])
    
    r10_mean = grp['r10'].mean()
    r10_ci = calc_bootstrap_ci(grp['r10'])
    
    rr10_mean = grp['rr10'].mean()
    rr10_ci = calc_bootstrap_ci(grp['rr10'])
    
    hit5 = grp['hit5'].mean()
    hit10 = grp['hit10'].mean()
    zero_res = int(grp['zero_result'].sum())
    lat_mean = grp['latency_ms'].mean()
    
    display_name = f"{m} ({cfg})"
    if m == 'like_baseline':
        display_name = "LIKE Baseline (e-Arsip)"
    
    main_table_rows.append({
        'Metode (Konfigurasi)': display_name,
        'n Kueri': len(grp),
        'P@5': f"{p5_mean:.4f} [{p5_ci[0]:.4f}, {p5_ci[1]:.4f}]",
        'R@10': f"{r10_mean:.4f} [{r10_ci[0]:.4f}, {r10_ci[1]:.4f}]",
        'MRR@10': f"{rr10_mean:.4f} [{rr10_ci[0]:.4f}, {rr10_ci[1]:.4f}]",
        'Hit@5': round(hit5, 4),
        'Hit@10': round(hit10, 4),
        'Nol Hasil (Kueri)': zero_res,
        'Rata2 Latensi (ms)': round(lat_mean, 2)
    })

df_main_summary = pd.DataFrame(main_table_rows)
print(f"=== TABEL 1: HASIL KINERJA TEMU KEMBALI KELOMPOK TEST (n = {n_test_queries} kueri) ===")
display(df_main_summary)

In [ ]:
# =====================================================================
# SEL 6: UJI STATISTIK KOMPARATIF BERPASANGAN (WILCOXON SIGNED-RANK TEST)
# Menguji signifikansi perbedaan MRR@10 antara metode terbaik vs LIKE Baseline
# =====================================================================
from scipy.stats import wilcoxon

# Tentukan metode fuzzy terbaik berdasarkan MRR@10 rata-rata pada test set
mean_mrr_by_model = df_test.groupby(['method', 'config'])['rr10'].mean().sort_values(ascending=False)
best_model_tuple = mean_mrr_by_model.index[0] if mean_mrr_by_model.index[0][0] != 'like_baseline' else mean_mrr_by_model.index[1]

m_best, cfg_best = best_model_tuple
series_best = df_test[(df_test['method'] == m_best) & (df_test['config'] == cfg_best)].sort_values('query_id')['rr10'].values
series_like = df_test[(df_test['method'] == 'like_baseline')].sort_values('query_id')['rr10'].values

diff = series_best - series_like
n_diff = len(diff)

print(f"Perbandingan Berpasangan: {m_best} ({cfg_best}) vs LIKE Baseline")
print(f"Jumlah Pasangan Kueri (n) : {n_diff}")
print(f"Kueri dengan skor beda    : {np.count_nonzero(diff)}")

if np.all(diff == 0):
    print("Statistik: Tidak ada perbedaan skor antar kedua metode.")
else:
    try:
        stat, p_value = wilcoxon(series_best, series_like, zero_method='wilcox')
        # Hitung Effect Size r = Z / sqrt(N)
        z_score = scipy.stats.norm.ppf(1.0 - p_value / 2.0) if p_value > 0 else 0.0
        effect_size_r = abs(z_score) / np.sqrt(n_diff) if n_diff > 0 else 0.0
        
        print(f"Wilcoxon W-Statistic      : {stat:.4f}")
        print(f"p-value (dua sisi)        : {p_value:.4e}")
        print(f"Effect Size (r)           : {effect_size_r:.4f}")
    except Exception as e:
        print(f"Catatan komputasi Wilcoxon: {e}")

print("\n[CATATAN ILMIAH] Karena ukuran sampel kelompok test tergolong terbatas (n=28),")
print("interpretasi uji signifikansi harus diposisikan sebagai indikasi empiris, bukan bukti kepastian populasi.")

In [ ]:
# =====================================================================
# SEL 7: DAMPAK PENYETELAN BOBOT (EQUAL WEIGHTS VS TUNED WEIGHTS)
# Bagian A: Evaluasi pada Kelompok Test | Bagian B: Hasil pada Kelompok Tuning
# =====================================================================
# Bagian A: Kelompok Test
fuzzy_methods = [m for m in df_test['method'].unique() if m != 'like_baseline']
weight_impact_rows = []

for m in fuzzy_methods:
    eq_mrr = df_test[(df_test['method'] == m) & (df_test['config'] == 'equal')]['rr10'].mean()
    tu_mrr = df_test[(df_test['method'] == m) & (df_test['config'] == 'tuned')]['rr10'].mean()
    eq_p5 = df_test[(df_test['method'] == m) & (df_test['config'] == 'equal')]['p5'].mean()
    tu_p5 = df_test[(df_test['method'] == m) & (df_test['config'] == 'tuned')]['p5'].mean()
    
    weight_impact_rows.append({
        'Metode Fuzzy': m,
        'Equal MRR@10': round(eq_mrr, 4),
        'Tuned MRR@10': round(tu_mrr, 4),
        'Selisih MRR': round(tu_mrr - eq_mrr, 4),
        'Equal P@5': round(eq_p5, 4),
        'Tuned P@5': round(tu_p5, 4),
        'Selisih P@5': round(tu_p5 - eq_p5, 4)
    })

print("=== TABEL 2A: DAMPAK PENYETELAN BOBOT PADA KELOMPOK TEST (EVALUASI AKHIR) ===")
display(pd.DataFrame(weight_impact_rows))

# Bagian B: Kelompok Tuning (Diambil dari tuning_results.csv)
print("\n=== TABEL 2B: RINGKASAN BOBOT OPTIMAL PADA KELOMPOK TUNING (OPTIMASI BOBOT) ===")
best_tuning_per_matcher = df_tuning.sort_values('mrr_at_10', ascending=False).groupby('matcher').first().reset_index()
display(best_tuning_per_matcher[['matcher', 'summary_weight', 'spm_weight', 'party_weight', 'mrr_at_10', 'precision_at_5']])

In [ ]:
# =====================================================================
# SEL 8: PERBANDINGAN PERFORMA: SUMBER KUERI NYATA VS SINTETIS
# Pelaporan terpisah sesuai standar Information Retrieval
# =====================================================================
source_counts = df_test.groupby('source')['query_id'].nunique()
print("Distribusi Sumber Kueri Uji:")
for src, count in source_counts.items():
    print(f"  - Sumber '{src}': {count} kueri")

df_source_eval = df_test.groupby(['source', 'method', 'config']).agg({
    'rr10': 'mean',
    'p5': 'mean',
    'zero_result': 'sum',
    'query_id': 'count'
}).reset_index()

df_source_eval.columns = ['Sumber Kueri', 'Metode', 'Konfigurasi', 'MRR@10', 'P@5', 'Nol Hasil', 'n Kueri']
df_source_eval['MRR@10'] = df_source_eval['MRR@10'].round(4)
df_source_eval['P@5'] = df_source_eval['P@5'].round(4)

print("\n=== TABEL 3: REKAPITULASI KINERJA BERDASARKAN SUMBER KUERI (TEST SPLIT) ===")
display(df_source_eval)

n_real = source_counts.get('nyata', 0)
if n_real < 3:
    print(f"\n[CATATAN KETERBATASAN]: Kueri bersumber kasus nyata berjumlah {n_real} kueri (< 3 kueri).")
    print("Pengujian didominasi oleh kueri sintetis yang dirancang terstruktur merefleksikan variasi penulisan arsiparis.")

In [ ]:
# =====================================================================
# SEL 9: RINCIAN PER JENIS KUERI & GANGGUAN PENGETIKAN (STRATIFIED DIAGNOSTIC)
# Label: Gambaran diagnostik eksploratif, bukan kesimpulan inferensial mutlak
# =====================================================================
print("=== GAMBARAN DIAGNOSTIK KINERJA BERDASARKAN RAGAM GANGGUAN PENGETIKAN ===")
print("(Catatan: Tabel ini memberikan gambaran pola, bukan kesimpulan inferensial mutlak)\n")

# Pilih metode perwakilan untuk dibandingkan: LIKE Baseline vs Tuned Matchers
rep_models = ['like_baseline', 'token_set', 'jaro_winkler', 'levenshtein']
df_rep = df_test[(df_test['method'].isin(rep_models)) & (df_test['config'].isin(['baseline', 'tuned']))]

noise_summary = df_rep.pivot_table(
    index='noise_type',
    columns='method',
    values='rr10',
    aggfunc='mean'
).round(4)

# Tambahkan kolom jumlah kueri n per jenis gangguan
noise_counts = df_test[df_test['method'] == 'like_baseline'].groupby('noise_type')['query_id'].count()
noise_summary.insert(0, 'n Kueri', noise_counts)

display(noise_summary)

In [ ]:
# =====================================================================
# SEL 10: AUDIT AKURASI EKSTRAKSI METADATA (WILSON SCORE 95% CI)
# Sumber: audit_summary.json
# =====================================================================
with open(BUNDLE_DIR / "audit_summary.json", "r", encoding="utf-8") as f:
    audit_data = json.load(f)

total_audit = audit_data.get('total_evaluated', 100)
fields_audit = audit_data.get('fields', {})

audit_rows = []
for fld_name, metrics in fields_audit.items():
    p = metrics.get('ketepatan', 0.0) * 100.0
    p_ci = metrics.get('wilson_ci_ketepatan', [0.0, 0.0])
    r = metrics.get('keterambilan', 0.0) * 100.0
    r_ci = metrics.get('wilson_ci_keterambilan', [0.0, 0.0])
    
    audit_rows.append({
        'Field Metadata': fld_name.upper(),
        'Sampel Dievaluasi': metrics.get('total_evaluated', total_audit),
        'Benar': metrics.get('n_benar', 0),
        'Salah': metrics.get('n_salah', 0),
        'Terlewat': metrics.get('n_terlewat', 0),
        'Ketepatan (Precision)': f"{p:.1f}% [{p_ci[0]*100:.1f}%, {p_ci[1]*100:.1f}%]",
        'Keterambilan (Recall)': f"{r:.1f}% [{r_ci[0]*100:.1f}%, {r_ci[1]*100:.1f}%]"
    })

df_audit_table = pd.DataFrame(audit_rows)
print(f"=== TABEL 4: AKURASI EKSTRAKSI METADATA DENGAN WILSON 95% CI (n = {total_audit} sampel) ===")
display(df_audit_table)

In [ ]:
# =====================================================================
# SEL 11: RELIABILITAS PENILAIAN RELEVANSI (INTER-RATER AGREEMENT)
# Sumber: agreement_summary.json
# =====================================================================
with open(BUNDLE_DIR / "agreement_summary.json", "r", encoding="utf-8") as f:
    agreement_data = json.load(f)

print("=== STATUS RELIABILITAS ANOTASI RELEVANSI ===")
status_agr = agreement_data.get('status', 'unknown')
n_samples = agreement_data.get('n_samples', 0)

print(f"Status Penilaian       : {status_agr}")
print(f"Jumlah Sampel Anotasi  : {n_samples} baris")
print(f"Persentase Kesepakatan : {agreement_data.get('percentage_agreement')}")
print(f"Kesepakatan Kelas Rel. : {agreement_data.get('relevant_class_agreement')}")
print(f"Cohen's Kappa          : {agreement_data.get('cohen_kappa')}")
print(f"Catatan Prosedural     : {agreement_data.get('notes')}")

In [ ]:
# =====================================================================
# SEL 12: GRAFIK PUBLIKASI ILMIAH (300 DPI) & EKSPOR TABEL CSV
# Menyertakan jumlah sampel n pada judul/label grafik
# =====================================================================
export_dir = Path("output/colab_export")
export_dir.mkdir(parents=True, exist_ok=True)

# 1. Ekspor tabel CSV utama
df_main_summary.to_csv(export_dir / "tabel_evaluasi_utama_test.csv", index=False)
print(f"[OK] Tabel utama diekspor ke: {export_dir / 'tabel_evaluasi_utama_test.csv'}")

# 2. Grafik Perbandingan Kinerja Utama
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=300)

# Subplot 1: MRR@10 dan Precision@5
methods_plot = ['LIKE Baseline', 'Levenshtein (Tuned)', 'Jaro-Winkler (Tuned)', 'Token-Set (Tuned)', 'N-Gram (Tuned)']
keys_plot = [('like_baseline', 'baseline'), ('levenshtein', 'tuned'), ('jaro_winkler', 'tuned'), ('token_set', 'tuned'), ('ngram', 'tuned')]

mrr_plot_vals = [df_test[(df_test['method'] == k[0]) & (df_test['config'] == k[1])]['rr10'].mean() * 100 for k in keys_plot]
p5_plot_vals = [df_test[(df_test['method'] == k[0]) & (df_test['config'] == k[1])]['p5'].mean() * 100 for k in keys_plot]

x = np.arange(len(methods_plot))
w = 0.35
ax1.bar(x - w/2, mrr_plot_vals, w, label='MRR@10 (x100)', color='#1f77b4', edgecolor='black', alpha=0.85)
ax1.bar(x + w/2, p5_plot_vals, w, label='P@5 (%)', color='#2ca02c', edgecolor='black', alpha=0.85)
ax1.set_title(f'Kinerja Temu Kembali Dokumen (n = {n_test_queries} kueri uji)')
ax1.set_ylabel('Skor Persentase (%)')
ax1.set_xticks(x)
ax1.set_xticklabels(methods_plot, rotation=25, ha='right', fontsize=9)
ax1.legend(loc='upper right')

# Subplot 2: Kueri Nol Hasil (Zero-Result)
zero_vals = [df_test[(df_test['method'] == k[0]) & (df_test['config'] == k[1])]['zero_result'].sum() for k in keys_plot]
colors_zero = ['#d62728' if z > 0 else '#2ca02c' for z in zero_vals]
bars2 = ax2.bar(methods_plot, zero_vals, color=colors_zero, edgecolor='black', alpha=0.85)
ax2.set_title(f'Angka Kegagalan Temu Kembali / Nol Hasil (Total = {n_test_queries} kueri)')
ax2.set_ylabel('Jumlah Kueri Nol Hasil')
ax2.set_xticklabels(methods_plot, rotation=25, ha='right', fontsize=9)
for bar in bars2:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, y + 0.3, str(int(y)), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
fig_path = export_dir / "fig_perbandingan_kinerja_300dpi.png"
fig.savefig(fig_path, dpi=300)
print(f"[OK] Grafik 300 DPI disimpan ke: {fig_path}")
plt.show()

In [ ]:
# =====================================================================
# SEL 13: GENERASI RINGKASAN EKSEKUTIF OTOMATIS DARI ANGKA (TEMPLAT RESMI)
# Aturan: Seluruh kalimat dibangkitkan dari variabel angka, tanpa pengetikan manual
# =====================================================================
like_mrr = df_test[df_test['method'] == 'like_baseline']['rr10'].mean()
like_zero = int(df_test[df_test['method'] == 'like_baseline']['zero_result'].sum())

# Cari metode fuzzy tuned terbaik
df_tuned = df_test[df_test['config'] == 'tuned']
best_tuned_name = df_tuned.groupby('method')['rr10'].mean().idxmax()
best_tuned_mrr = df_tuned.groupby('method')['rr10'].mean().max()
best_tuned_zero = int(df_test[(df_test['method'] == best_tuned_name) & (df_test['config'] == 'tuned')]['zero_result'].sum())
best_tuned_p5 = df_test[(df_test['method'] == best_tuned_name) & (df_test['config'] == 'tuned')]['p5'].mean()

mrr_best_ci = calc_bootstrap_ci(df_test[(df_test['method'] == best_tuned_name) & (df_test['config'] == 'tuned')]['rr10'])
mrr_like_ci = calc_bootstrap_ci(df_test[df_test['method'] == 'like_baseline']['rr10'])

pct_like_zero = (like_zero / n_test_queries) * 100.0
pct_best_zero = (best_tuned_zero / n_test_queries) * 100.0

template_summary = f"""
RINGKASAN EKSEKUTIF TEMUAN PENELITIAN (DIBANGKITKAN SECARA OTOMATIS)
----------------------------------------------------------------------
1. Perbandingan Kinerja Utama:
   Pada kelompok uji (test split, n = {n_test_queries} kueri), metode terbaik hasil penyetelan 
   bobot adalah '{best_tuned_name} (tuned)' dengan nilai MRR@10 sebesar {best_tuned_mrr:.4f} 
   (95% CI: [{mrr_best_ci[0]:.4f}, {mrr_best_ci[1]:.4f}]) dan Precision@5 sebesar {best_tuned_p5:.4f}, 
   dibandingkan LIKE Baseline yang memperoleh MRR@10 sebesar {like_mrr:.4f} 
   (95% CI: [{mrr_like_ci[0]:.4f}, {mrr_like_ci[1]:.4f}]).

2. Pengentasan Masalah Kegagalan Temu Kembali (Zero-Result Problem):
   Sistem eksisting (LIKE Baseline) mengalami kegagalan temu kembali mutlak pada 
   {like_zero} dari {n_test_queries} kueri ({pct_like_zero:.1f}%), sedangkan metode '{best_tuned_name} (tuned)' 
   berhasil menekan angka nol hasil menjadi {best_tuned_zero} kueri ({pct_best_zero:.1f}%).

3. Penyetelan Bobot Multi-Field:
   Pemberian bobot spesifik per field terbukti menyesuaikan karakteristik kueri 
   antara entitas formal (Nomor SPM) dan deskripsi kegiatan (Ringkasan).
"""

print(template_summary)

---
## ⚠️ Keterbatasan Penelitian (Academic Limitations)

Penyajian hasil evaluasi pada penelitian ini wajib dimaknai dalam batas-batas metodologis berikut:
1. **Dominasi Kueri Sintetis:** Sebagian besar kueri uji merupakan kueri sintetis terstruktur yang dibangkitkan untuk mensimulasikan ragam kesalahan pengetikan manusia nyata (*noise models*).
2. **Anotasi Relevansi Tunggal:** Penilaian relevansi utama (1.077 pasang) diselesaikan oleh peneliti, sementara verifikasi ganda arsiparis independen masih berstatus sampel verifikasi.
3. **Skala Pengguna Terbatas:** Uji coba belum melibatkan lebih dari 5 pengguna akhir simultan di lingkungan operasional dinas.
4. **Salinan Data Statis:** Pengujian menggunakan salinan *dump* statis 1.737 arsip tahun 2021-2023, bukan pengujian *live streaming* basis data produksi e-Arsip yang terus bertambah.

---
## 🔒 Prosedur Penutupan Sesi & Keamanan Runtime

Sebelum menutup sesi Google Colab atau mengunggah perubahan ke repositori:
1. **Bersihkan Seluruh Keluaran Sel (*Clear All Outputs*):**
   Pilih menu `Edit` $\to$ `Clear all outputs` agar tidak ada angka atau grafik yang tertinggal di berkas JSON `.ipynb` secara permanen.
2. **Putuskan Runtime (*Disconnect & Delete Runtime*):**
   Pilih menu panah bawah di pojok kanan atas Colab $\to$ pilih **`Disconnect and delete runtime`** untuk menghapus seluruh berkas sementara di lingkungan *cloud*.
3. **Jaga Kerahasiaan:**
   Jangan membagikan tautan notebook atau folder Drive secara publik.